# LLM03 Supply Chain Vulnerabilities — Upload Artifacts & Run Evaluation

**OWASP Category**: LLM03 — Supply Chain Vulnerabilities | **Risk Severity**: High

This notebook:
1. **Uploads** all artifact files (scenarios, checks) from the local folder structure and registers them in Okareo.
2. **Runs** the full LLM03 supply chain test suite against your target AI agent.

Uploading is **idempotent** — re-running will not create duplicates (scenarios return existing, checks upsert).
Registered IDs are available in-memory for the evaluation steps below.

**Scenarios**:
- **Scenario 1**: Third-party model behavioral validation (model-based check)
- **Scenario 2**: Dependency and provenance integrity verification (code-based check)

In [ ]:
%pip install okareo python-dotenv --quiet

In [ ]:
import sys
import json
from pathlib import Path

# Add project root for owasp.common import
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))

from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver

from owasp.common import (
    init_okareo,
    parse_artifact,
    CodeCheckFromSource,
    build_target,
    SINGLE_TURN_DRIVER_TEMPLATE,
)

okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")
print(f"Category directory: {CATEGORY_DIR}")


---
## Part 1 — Upload Artifacts

### Upload Scenarios

Scans `scenarios/` for `.jsonl` files and uploads each via `upload_scenario_set`.

In [ ]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}

for jsonl_path in sorted(scenarios_dir.glob("*.jsonl")):
    scenario_name = f"LLM03-{jsonl_path.stem}"
    print(f"Uploading scenario: {scenario_name} from {jsonl_path.name}")

    scenario = okareo.upload_scenario_set(
        scenario_name=scenario_name,
        file_path=str(jsonl_path),
    )
    registered_scenarios[scenario_name] = scenario
    print(f"  ✓ Registered: {scenario_name} (ID: {scenario.scenario_id})")

print(f"\nTotal scenarios uploaded: {len(registered_scenarios)}")

### Register Checks

Registers both check types:
- **Model-based** (`.md`): Parsed from YAML front matter + prompt template body
- **Code-based** (`.py`): Read as raw source and registered via `CodeBasedCheck`

In [ ]:
checks_dir = CATEGORY_DIR / "checks"
registered_checks = {}

for md_path in sorted(checks_dir.glob("*.md")):
    check_data = parse_artifact(md_path)
    print(f"Registering model-based check: {check_data['name']} from {md_path.name}")

    check_obj = ModelBasedCheck(
        prompt_template=check_data["prompt_template"],
        check_type=CheckOutputType.PASS_FAIL,
    )
    result = okareo.create_or_update_check(
        name=check_data["name"],
        description=check_data["description"],
        check=check_obj,
    )
    registered_checks[check_data["name"]] = {"id": result.id, "type": "model"}
    print(f"  ✓ Registered: {check_data['name']} (ID: {result.id})")

for py_path in sorted(checks_dir.glob("*.py")):
    data = parse_artifact(py_path)
    print(f"Registering code-based check: {data['name']} from {py_path.name}")

    check_obj = CodeCheckFromSource(code_contents=data["code_contents"])
    result = okareo.create_or_update_check(
        name=data["name"],
        description=data["description"],
        check=check_obj,
    )
    registered_checks[data["name"]] = {"id": result.id, "type": "code"}
    print(f"  ✓ Registered: {data['name']} (ID: {result.id})")

print(f"\nTotal checks registered: {len(registered_checks)}")


### Artifact Upload Summary

In [ ]:
print("=" * 60)
print("LLM03 Supply Chain — Artifact Upload Summary")
print("=" * 60)
print(f"\nScenarios ({len(registered_scenarios)}):")
for name, sc in registered_scenarios.items():
    print(f"  • {name} → {sc.scenario_id}")
print(f"\nChecks ({len(registered_checks)}):")
for name, info in registered_checks.items():
    print(f"  • {name} [{info['type']}] → {info['id']}")
print("\n✓ All artifacts ready. Proceeding to evaluation...")

---
## Part 2 — Run Evaluation

### Configuration

The target agent is loaded from the shared `owasp/target.json` file.
All OWASP category notebooks reference this same file so that every control evaluates the same agent.

Both scenarios use `okareo.run_simulation()` with `max_turns=1` (single-turn):
- **Scenario 1** (behavioral validation): Model-based check evaluates the agent's response for undocumented capabilities.
- **Scenario 2** (provenance integrity): Code-based check deterministically validates structured metadata.

In [ ]:
# Target loaded from owasp/target.json. To use a different config: target = build_target(CATEGORY_DIR, config_path="target.prod.json")
target = build_target(CATEGORY_DIR)
TARGET_NAME = target.name
print(f"✓ Target agent: {TARGET_NAME}")

BEHAVIORAL_CHECK = "LLM03-behavioral-boundary-detector"
PROVENANCE_CHECK = "LLM03-provenance-integrity-check"

### Scenario 1 — Third-Party Model Behavioral Validation

Sends probing prompts via a pass-through driver to test whether the model exhibits
undocumented capabilities. Evaluated by the model-based `behavioral-boundary-detector` check.

In [ ]:
BEHAVIORAL_SCENARIO = "LLM03-behavioral-validation"

single_turn_driver = Driver(
    temperature=0,
    name=f"{TARGET_NAME}-single-turn-driver",
    prompt_template=SINGLE_TURN_DRIVER_TEMPLATE,
)

print(f"\n{'='*60}")
print(f"Running: {BEHAVIORAL_SCENARIO}")
print(f"Check: {BEHAVIORAL_CHECK} (model-based)")
print(f"{'='*60}")

behavioral_result = None
try:
    scenario = registered_scenarios[BEHAVIORAL_SCENARIO]

    behavioral_result = okareo.run_simulation(
        target=target,
        driver=single_turn_driver,
        name=f"LLM03 Eval — {BEHAVIORAL_SCENARIO}",
        api_key=OKAREO_API_KEY,
        first_turn="driver",
        scenario=scenario,
        max_turns=1,
        checks=[BEHAVIORAL_CHECK],
    )
    print(f"  ✓ Test run complete: {behavioral_result.id}")
    if hasattr(behavioral_result, "app_link") and behavioral_result.app_link:
        print(f"  View: {behavioral_result.app_link}")
except Exception as e:
    print(f"  ✗ Error: {e}")

### Scenario 2 — Dependency and Provenance Integrity

Sends structured metadata payloads and validates them deterministically using the
code-based `provenance-integrity-check`. The check operates on the scenario input
metadata, not the agent's response.

In [ ]:
PROVENANCE_SCENARIO = "LLM03-provenance-integrity"

print(f"\n{'='*60}")
print(f"Running: {PROVENANCE_SCENARIO}")
print(f"Check: {PROVENANCE_CHECK} (code-based)")
print(f"{'='*60}")

provenance_result = None
try:
    scenario = registered_scenarios[PROVENANCE_SCENARIO]

    provenance_result = okareo.run_simulation(
        target=target,
        driver=single_turn_driver,
        name=f"LLM03 Eval — {PROVENANCE_SCENARIO}",
        api_key=OKAREO_API_KEY,
        first_turn="driver",
        scenario=scenario,
        max_turns=1,
        checks=[PROVENANCE_CHECK],
    )
    print(f"  ✓ Test run complete: {provenance_result.id}")
    if hasattr(provenance_result, "app_link") and provenance_result.app_link:
        print(f"  View: {provenance_result.app_link}")
except Exception as e:
    print(f"  ✗ Error: {e}")

### Results Summary

In [ ]:
print("\n" + "=" * 60)
print("LLM03 SUPPLY CHAIN VULNERABILITIES — EVALUATION RESULTS")
print("OWASP Category: LLM03 | Risk Severity: High")
print("=" * 60)

all_results = {
    BEHAVIORAL_SCENARIO: behavioral_result,
    PROVENANCE_SCENARIO: provenance_result,
}

print(f"\n{'Scenario':<42} {'Check Type':<12} {'Status':<10} {'Link / Run ID'}")
print("-" * 110)

check_types = {
    BEHAVIORAL_SCENARIO: "model",
    PROVENANCE_SCENARIO: "code",
}

for name, result in all_results.items():
    ctype = check_types.get(name, "unknown")
    if result is None:
        print(f"{name:<42} {ctype:<12} {'ERROR':<10} N/A")
    else:
        link = getattr(result, "app_link", None) or result.id
        print(f"{name:<42} {ctype:<12} {'COMPLETE':<10} {link}")

errors = sum(1 for r in all_results.values() if r is None)
print(f"\nTotal evaluated: {len(all_results)} | Errors: {errors}")
if not errors:
    print("✓ All scenarios completed. See Okareo dashboard for full results.")

### Detailed Results (Optional)

Retrieve per-row scores and model outputs for any completed test run.

In [ ]:
# Uncomment to inspect a specific completed run in detail:
# RUN_ID = "<paste_run_id_here>"
# detailed = okareo.get_test_run(RUN_ID)
# for row in (detailed.model_results or []):
#     print(f"Input:   {str(row.get('scenario_input', ''))[:80]}")
#     print(f"Output:  {str(row.get('model_output', ''))[:80]}")
#     print(f"Checks:  {row.get('check_scores', {})}")
#     print("-" * 40)